In [2]:
# UCS
from tkinter import *
from queue import PriorityQueue
import copy
import random

# random goal
nums = list(range(9))
random.shuffle(nums)

goal = [
    nums[0:3],
    nums[3:6],
    nums[6:9]
]


def zero(state):

    for i in range(3):
        for j in range(3):

            if state[i][j] == 0:
                return i, j


def tap_con(state):

    x, y = zero(state)

    moves = []

    if x > 0:
        moves.append((x-1, y))

    if x < 2:
        moves.append((x+1, y))

    if y > 0:
        moves.append((x, y-1))

    if y < 2:
        moves.append((x, y+1))

    result = []

    for nx, ny in moves:

        new = copy.deepcopy(state)

        new[x][y], new[nx][ny] = \
        new[nx][ny], new[x][y]

        result.append(new)

    return result


# hàm tính chi phí:
# số ô sai so với goal
def cost(state):

    wrong = 0

    for i in range(3):
        for j in range(3):

            if state[i][j] != goal[i][j]:
                wrong += 1

    return wrong


# random start theo goal
start = copy.deepcopy(goal)

for _ in range(10):

    start = random.choice(tap_con(start))


def ucs(start):

    frontier = PriorityQueue()

    # (chi phí, path)
    frontier.put((cost(start), [start]))

    reached = set()

    while not frontier.empty():

        current_cost, path = frontier.get()

        state = path[-1]

        if state == goal:
            return path

        reached.add(str(state))

        for con in tap_con(state):

            if (str(con) not in reached and
                con not in [p[1][-1] for p in frontier.queue]):

                new_cost = cost(con)

                frontier.put(
                    (new_cost, path + [con])
                )

    return None

#GUI
root = Tk()

root.title("8 Puzzle UCS")

buttons = []


def draw(state):

    for i in range(3):
        for j in range(3):

            value = state[i][j]

            if value == 0:
                text = ""
            else:
                text = str(value)

            buttons[i][j]["text"] = text


def animate(path, step=0):

    if step < len(path):

        draw(path[step])

        root.after(500, animate, path, step + 1)


def solve():

    path = ucs(start)

    if path:
        animate(path)


for i in range(3):

    row = []

    for j in range(3):

        btn = Button(
            root,
            width=5,
            height=2,
            font=("Arial", 24)
        )

        btn.grid(row=i, column=j)

        row.append(btn)

    buttons.append(row)


solve_btn = Button(
    root,
    text="Solve UCS",
    command=solve,
    font=("Arial", 16)
)

solve_btn.grid(
    row=3,
    column=0,
    columnspan=3,
    sticky="we"
)

draw(start)

print("START:\n")
for row in start:
    print(row)

print("\nGOAL:\n")
for row in goal:
    print(row)

root.mainloop()

START:

[2, 6, 4]
[1, 3, 5]
[7, 8, 0]

GOAL:

[0, 6, 4]
[2, 1, 5]
[7, 3, 8]
